#Importacion Librerias


In [0]:
from pyspark.sql import functions as F
from datetime import datetime

#Creando Funciones de Apoyo

In [0]:
def get_current_date():
    """
    Get current date
    Returns:
        Current date in yyyy-MM-dd HH:mm:ss.SSS format
    """
    return spark.sql(
        "select date_format(current_timestamp,'yyyy-MM-dd HH:mm:ss.SSS')"
    ).collect()[0][0]



#Invocacion a tabla en Bronze

In [0]:
#obtener los datos de la tabla bronse
df_bronze = spark.read.table("workspace.yelp_bronze.business")

#Obtenemos la hora de ejecucion
date = get_current_date()
type_proces ='Full'

#Transformacion Datos

In [0]:
#Lectura de los campos requeridos
df_silver = df_bronze.select(
    F.col('business_id').alias('business_id'),
    F.col('name').alias('nombre'),
    F.col('city').alias('ciudad'),
    F.col('state').alias('estado'),
    F.col('latitude').alias('latitud'),
    F.col('longitude').alias('longitud'),
    F.col('stars').alias('estrellas'),
    F.col('review_count').cast('int').alias('n_resenas'),
    F.when(F.col('is_open')=='1',True).otherwise(False).alias('esta_abierto'),
    #F.col('is_open').alias('esta_abierto'),
    F.col('attributes').alias('categorias'),
    F.lit(date).cast('timestamp').alias("fecha_carga")
)


#Grabando en tabla Silver

In [0]:

print(f'EL proceso de grabacion es : {type_proces}')
if type_proces =='Full' :
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.yelp_silver.slv_business")
else :
    df_silver.write.format("delta").mode("append").option("overwriteSchema", "true").saveAsTable("workspace.yelp_silver.slv_business")


In [0]:
dbutils.notebook.exit('SUCCESS')

In [0]:
display(df_silver)

In [0]:
display(df_bronze)